# Phase 1 — Environment Setup
**Code-Mixed NLP Failure Analysis**

Run every cell top to bottom. Do not proceed to Phase 2 until the final verification cell shows **OK** for the models you intend to use.

Prereq (Phase 0): you already have real sentences + labels on disk. This notebook only sets up the environment and confirms all models load.

## 1.1  Confirm GPU
Runtime → Change runtime type → Hardware accelerator → **T4 GPU** first, then run this.

In [ ]:
!nvidia-smi

## 1.2  Install libraries
`transformers>=4.41` floor is deliberate: the `eval_strategy=` argument used in Phase 2 does not exist under that name in older releases (it was `evaluation_strategy`). Pinning the floor here means an old Colab base image fails *now*, loudly, instead of mid-training.

In [ ]:
# sentencepiece is required: IndicBERT v1 (ALBERT) uses a SentencePiece tokenizer.
!pip install -q "transformers>=4.41" datasets evaluate seqeval accelerate sentencepiece
!pip install -q scikit-learn scipy statsmodels pandas matplotlib seaborn

### Record exact versions
Phase 6 (reproducibility) needs these. Copy the printed block into a `requirements.txt` in your repo now, while it's in front of you.

In [ ]:
import transformers, datasets, evaluate, sklearn, scipy, statsmodels, torch, numpy, pandas
for m in [transformers, datasets, evaluate, sklearn, scipy, statsmodels, torch, numpy, pandas]:
    print(f"{m.__name__}=={m.__version__}")

## 1.3  Mount Drive and create the project folder structure
Everything downstream (checkpoints, per-sentence prediction CSVs, feature files) gets saved here, **not** to Colab's temporary disk. Folders are created up front so Phase 2/3 outputs have a home.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/codemix_project'
for sub in ['', 'data', 'checkpoints', 'results', 'preds', 'features', 'figures']:
    os.makedirs(os.path.join(PROJECT, sub), exist_ok=True)

print('Project root:', PROJECT)
print('Subfolders  :', sorted(os.listdir(PROJECT)))

## 1.4a  Hugging Face login (needed for the gated IndicBERT)
`ai4bharat/indic-bert` is gated: visit its model page while logged in, click **Agree**, then create a **read** token at huggingface.co/settings/tokens and paste it below. The other four models are ungated — if you skip IndicBERT you can skip this cell.

In [ ]:
from huggingface_hub import login
# Prefer Colab's Secrets (🔑 icon in the left sidebar): add a secret named HF_TOKEN,
# then this reads it without pasting the token into the notebook.
try:
    from google.colab import userdata
    login(userdata.get('HF_TOKEN'))
    print('Logged in via Colab secret HF_TOKEN.')
except Exception as e:
    print('No HF_TOKEN secret found. Either add one, or uncomment the line below and paste a token:')
    print(' ', e)
    # login('hf_xxxxxxxxxxxxxxxxx')

## 1.4b  Verify ALL five models load
This is the real gate before Phase 2. Each model's tokenizer + a sequence-classification head are loaded and one Hinglish sentence is tokenized. The `num_labels=3` head is randomly initialised on purpose — the "newly initialized" warning is expected and harmless for a load test.

Read the summary table at the end: every model you plan to fine-tune must say **OK**.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import warnings, traceback
warnings.filterwarnings('ignore')

MODELS = {
    'mBERT':       'bert-base-multilingual-cased',
    'MuRIL':       'google/muril-base-cased',
    'IndicBERT':   'ai4bharat/indic-bert',          # gated + ALBERT architecture
    'HingBERT':    'l3cube-pune/hing-bert',
    'HingRoBERTa': 'l3cube-pune/hing-roberta',
}

SENTENCE = 'kal movie dekhi it was amazing yaar'
results = {}

for name, hf_id in MODELS.items():
    try:
        tok = AutoTokenizer.from_pretrained(hf_id)
        _ = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=3)
        toks = tok.tokenize(SENTENCE)
        results[name] = ('OK', f'{len(toks)} tokens')
        print(f'[OK]   {name:12s} {hf_id}')
        print(f'       tokens: {toks}\n')
    except Exception as e:
        msg = f'{type(e).__name__}: {str(e)[:120]}'
        results[name] = ('FAIL', msg)
        print(f'[FAIL] {name:12s} {hf_id}')
        print(f'       {msg}\n')

print('=' * 60)
print('SUMMARY')
print('=' * 60)
for name, (status, info) in results.items():
    print(f'{status:5s}  {name:12s}  {info}')

failed = [n for n, (s, _) in results.items() if s == 'FAIL']
if failed:
    print(f'\n⚠  {failed} failed. If IndicBERT is the only failure, it is almost')
    print('   certainly the access gate — accept its terms + set HF_TOKEN (cell 1.4a).')
    print('   For others: search huggingface.co for the current model ID.')
else:
    print('\n✓  All models load. Environment is ready for Phase 2.')

## Done — before you leave Phase 1
- [ ] GPU shows up in `nvidia-smi`
- [ ] Version block copied into `requirements.txt`
- [ ] Drive folders created under `codemix_project/`
- [ ] All intended models say **OK** in the summary

Next: Phase 2.1 — load your Phase 0 data into a DataFrame with a persistent `sentence_id`. Everything in Phase 4 depends on that id surviving intact.